# 01. Ingest and Deduplicate

This notebook walks through one document end to end with the current `GraphRAG` step API: read, chunk, embed, persist, extract entities, inspect the graph, search it, find duplicate candidates, merge duplicates, and query the cleaned graph again.

`read_document(...)` is the only file-format detection step. It stores `source_format` and `content_format` in `document.metadata`, treats markdown files as markdown content, and converts PDFs to markdown in memory before chunking. If you ingest markdown already held in memory, use `ingest_text(..., format="markdown")`.

Sample data note:

- The tutorial texts in `notebooks/experimental_data/` are Medium.com articles by Filip Wojcik.
- Source profile: `https://medium.com/@filip.igor.wojcik`
- They are fully accessible without a subscription.

Install prerequisites before running the notebook:

```bash
uv sync --extra falkordblite --extra notebooks
```

You also need model credentials for the chat and embedding models you choose below.


In [ ]:
import os
from pathlib import Path

import autoroot  # noqa
from dotenv import load_dotenv
from rich import print

from grawiki.db import FalkorGraphDB
from grawiki.rag import GraphRAG

load_dotenv(override=True)


def locate_notebooks_dir() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / "experimental_data").exists():
        return cwd
    if (cwd / "notebooks" / "experimental_data").exists():
        return cwd / "notebooks"
    raise FileNotFoundError(
        "Could not locate the notebooks directory from the current working directory."
    )


NOTEBOOKS_DIR = locate_notebooks_dir()
DATA_DIR = NOTEBOOKS_DIR / "experimental_data"
DB_PATH = NOTEBOOKS_DIR / "local_falcor.db"
SOURCE_PATH = DATA_DIR / "deep_knowledge_from_deep_learning.txt"
MODEL = (
    os.getenv("GRAWIKI_MODEL")
    or os.getenv("LLM_MODEL")
    or "openrouter/openai/gpt-5-mini"
)
EMBEDDING_MODEL = (
    os.getenv("GRAWIKI_EMBEDDING_MODEL")
    or os.getenv("EMBEDDING_MODEL")
    or "openrouter:openai/text-embedding-3-small"
)

In [ ]:
database = FalkorGraphDB(db_path=DB_PATH, graph_name="grawiki_notebooks")
rag = GraphRAG(
    model=MODEL,
    embedding_model=EMBEDDING_MODEL,
    db=database,
    resolve_entities_on_ingest=True,
    kg_extractor_kwargs={"reasoning_effort": "minimal"}, # to reduce the entity resolution time
)

print("GraphRAG initialized with ingest-time entity resolution enabled.")

# One-shot ingestion with "ingest"

In [ ]:
await rag.ingest(SOURCE_PATH, show_progress=True)

## Step-by-step ingestion

The following cell uses the public notebook-friendly methods directly instead of the one-shot `ingest(...)` wrapper. That makes every intermediate artifact visible. Chunk routing is format-driven, so `rag.chunk_document(document)` reads `document.metadata["content_format"]` instead of inferring behavior from the source path.


In [ ]:
document = rag.read_document(SOURCE_PATH)
chunks = rag.chunk_document(document)
document_embedding = await rag.embed_document(document)
chunk_embeddings = await rag.embed_chunks(chunks)
document_node = rag.build_document_node(document, document_embedding)
chunk_nodes = rag.build_chunk_nodes(chunks, chunk_embeddings)
await rag.persist_document_and_chunks(document_node, chunk_nodes)
chunk_graphs = await rag.extract_kg_per_chunk(chunks, show_progress=True)
await rag.persist_entities_and_relationships(
    [chunk.id for chunk in chunks], chunk_graphs
)

print(
    {
        "document_id": document.id,
        "document_metadata": document.metadata,
        "chunk_count": len(chunks),
        "chunk_types": sorted({chunk.metadata.get("type", "text") for chunk in chunks}),
        "document_embedding_dims": len(document_embedding),
        "chunk_graph_count": len(chunk_graphs),
    }
)

# Check results

In [ ]:
document_rows = database.ro_query(
    "MATCH (n:__document__) RETURN n.id, n.name ORDER BY n.name LIMIT 3"
).result_set
chunk_rows = database.ro_query(
    "MATCH (n:__chunk__) RETURN n.id, n.name, n.document_id ORDER BY n.id LIMIT 5"
).result_set
entity_rows = database.ro_query(
    "MATCH (n:__entity__) RETURN n.id, n.name, n.semantic_key ORDER BY n.semantic_key, n.id LIMIT 10"
).result_set
entities = await database.list_entities()

print({"documents": document_rows, "chunks": chunk_rows, "entity_count": len(entities)})
print("First persisted entities:")
for entity in entities[:10]:
    print(
        {
            "id": entity.id,
            "name": entity.name,
            "labels": sorted(entity.labels),
            "semantic_key": entity.semantic_key,
        }
    )

In [ ]:
focus_entity = entities[0] if entities else None
if focus_entity is None:
    print("No entities were persisted.")
else:
    neighbors = await database.neighbor_relationships(
        node_ids=[focus_entity.id], limit_per_node=5
    )
    print({"focus_entity": focus_entity.name, "neighbors": neighbors[focus_entity.id]})

search_hits = await rag.search("Knowledge graphs", limit=5)
print("GraphRAG search hits:")
for hit in search_hits:
    print(
        {
            "score": round(hit.score, 4),
            "matched_on": hit.matched_on,
            "id": hit.node.id,
            "name": hit.node.name,
            "labels": sorted(hit.node.labels),
        }
    )

## Duplicate inspection

Use the duplicate finder first so the notebook shows the candidate groups before applying the destructive merge step. Exact counts may vary slightly across model runs.


In [ ]:
def merge_report_payload(report):
    return {
        "master_id": report.master_id,
        "duplicate_ids": list(report.duplicate_ids),
        "source": report.source,
        "merged_labels": list(report.merged_labels),
        "property_conflicts": report.property_conflicts,
    }


duplicate_candidates = await rag.find_entity_duplicate_candidates(
    limit=5, threshold=0.9
)
dry_run_reports = await rag.dedupe_entities(
    limit=5, threshold=0.9, min_merge_score=0.95, dry_run=True
)

print(
    {
        "semantic_key_collision_groups": len(
            duplicate_candidates.semantic_key_collision_candidates
        ),
        "similarity_candidate_groups": len(duplicate_candidates.similarity_candidates),
        "dry_run_merge_reports": [
            merge_report_payload(report) for report in dry_run_reports
        ],
    }
)

In [ ]:
applied_reports = await rag.dedupe_entities(
    limit=5, threshold=0.9, min_merge_score=0.95, dry_run=False
)
entities_after_dedupe = await database.list_entities()

print("Applied merge reports:")
for report in applied_reports:
    print(merge_report_payload(report))

print({"entity_count_after_dedupe": len(entities_after_dedupe)})
for entity in entities_after_dedupe[:10]:
    print(
        {
            "id": entity.id,
            "name": entity.name,
            "labels": sorted(entity.labels),
            "semantic_key": entity.semantic_key,
        }
    )

In [ ]:
final_hits = await rag.search("Knowledge graph concept", limit=5)
comparison_rows = database.ro_query(
    "MATCH (s:__entity__)-[r]->(t:__entity__) RETURN s.name, type(r), t.name ORDER BY s.name, type(r), t.name LIMIT 10"
).result_set

print("Final GraphRAG hits:")
for hit in final_hits:
    print(
        {
            "score": round(hit.score, 4),
            "matched_on": hit.matched_on,
            "name": hit.node.name,
            "labels": sorted(hit.node.labels),
            "content_preview": hit.node.properties.get("content", "")[:500],
        }
    )

print({"entity_relationship_samples": comparison_rows})

## Next steps

- Notebook 2 reuses the same FalkorDB graph for a `pydantic_ai.Agent` demo with `search`, `remember`, and `recall`.
- Notebook 3 builds a lightweight visualization over the populated graph.
- The other article files in `experimental_data/` are good follow-up ingestion exercises once this baseline flow works.


In [ ]:
# Run this once when you are finished with the notebook session.
database.close()